In [ ]:
# Installa el cliente de InfluxDB y polars para Python junto a paquetes necesarios
%pip install influxdb-client
%pip install polars

In [ ]:
# Importar polars como pl y
import polars as pl

In [ ]:
# Importa la librería de InfluxDB para Python
import influxdb_client

In [ ]:
# Crea un cliente de InfluxDB con la URL, token y organización especificados, si sale true significa que la conexión fue exitosa, si sale false significa que hubo un error en la conexión
from influxdb_client import InfluxDBClient

client = InfluxDBClient(
    url="http://localhost:8086",
    token="TU_TOKEN", # Aqui hay que poner el token del usuario
    org="TU_ORG" # Aqui hay que poner la organización del usuario
)

print(client.ping())

In [ ]:
# Crea el bucket 'Copia seguridad' si no existe todavia
from influxdb_client import InfluxDBClient

TOKEN  = "TU_TOKEN"# Aqui hay que poner el token del usuario
ORG    = "TU_ORG" # Aqui hay que poner la organización del usuario
URL    = "http://localhost:8086"
BUCKET = "Copia seguridad"

client = InfluxDBClient(url=URL, token=TOKEN, org=ORG)
buckets_api = client.buckets_api()

existing = buckets_api.find_bucket_by_name(BUCKET)
if existing:
    print(f"Bucket '{BUCKET}' ya existe ({existing.id})")
else:
    nuevo = buckets_api.create_bucket(bucket_name=BUCKET, org=ORG)
    print(f"Bucket '{BUCKET}' creado correctamente ({nuevo.id})")

client.close()

In [ ]:
# Lee el archivo CSV "steam_data.csv" y lo carga en un DataFrame de Polars
df = pl.read_csv("steam_data.csv")

In [ ]:
df = pl.read_csv(
    "steam_data.csv",
    separator=";",
    try_parse_dates=True
)

df_juegos = df.filter(
    pl.col("Game").str.contains("(?i)counter.?strike|among us")
)

print(df_juegos.select("Game").unique())
print(df_juegos.shape)
print(df_juegos.head(2))

In [ ]:
# Convierte la columna "Average Players" a tipo float64, eliminando las comas y reemplazándolas por nada
df_envio = df_juegos.with_columns(
    pl.col("Average Players")
      .str.replace_all(",", "")
      .cast(pl.Float64, strict=False)
)

In [ ]:
# Subir los datos a InfluxDB
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

client = InfluxDBClient(
    url="http://localhost:8086", # Aqui hay que poner la dirección de influxDB por defecto con el puerto 8086
    token="TU_TOKEN", # Aqui hay que poner el token de usuario
    org="TU_ORG", # Aqui hay que poner la organización del usuario
    timeout=60000
)

write_api = client.write_api(write_options=SYNCHRONOUS)

# Crear los puntos a partir del DataFrame filtrado de Polars
puntos = []

for fila in df_envio.iter_rows(named=True):
    punto = (
        Point("steam_data")
        .tag("game", fila["Game"])
        .time(fila["DateTime"], WritePrecision.US)
    )

    if fila["Players"] is not None:
        punto = punto.field("players", int(fila["Players"]))

    if fila["Average Players"] is not None:
        punto = punto.field("average_players", float(fila["Average Players"]))

    puntos.append(punto)

# Subirlos en lotes de 500
bucket = "Copia seguridad"
lote_tamano = 500

for inicio in range(0, len(puntos), lote_tamano):
    lote = puntos[inicio:inicio + lote_tamano]
    write_api.write(bucket=bucket, org="TU_ORG", record=lote) # Aqui poner la organización del usuario tambien
    print(f"Enviados {min(inicio + lote_tamano, len(puntos))} de {len(puntos)}")

client.close()